<a href="https://colab.research.google.com/github/Vishwas-Chaudhary/ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
print("""
METHOD CHOICE: Random Forest Regressor

I will use a Random Forest Regressor for this lane.

The goal is to estimate future content performance so that pages
can be ranked for review. Random Forest is suitable because it can
capture non-linear relationships between search exposure, clicks,
position, sessions, and engagement signals.

The model will use only information available at the March
decision moment. Future April outcome values will be used only
as the target, not as input features.

The main features are:
1. gsc_avg_position
2. gsc_impressions
3. gsc_clicks
4. ga4_sessions
5. scroll_events

The target is future April CTR, calculated from April clicks and
April impressions.

This model is intended for decision support. It does not establish
that any feature causes future performance.

The model will later be compared with the Week-4 rule-based
baseline using the same evaluation data and metric.
""")


METHOD CHOICE: Random Forest Regressor

I will use a Random Forest Regressor for this lane.

The goal is to estimate future content performance so that pages
can be ranked for review. Random Forest is suitable because it can
capture non-linear relationships between search exposure, clicks,
position, sessions, and engagement signals.

The model will use only information available at the March
decision moment. Future April outcome values will be used only
as the target, not as input features.

The main features are:
1. gsc_avg_position
2. gsc_impressions
3. gsc_clicks
4. ga4_sessions
5. scroll_events

The target is future April CTR, calculated from April clicks and
April impressions.

This model is intended for decision support. It does not establish
that any feature causes future performance.

The model will later be compared with the Week-4 rule-based
baseline using the same evaluation data and metric.



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
print("""
SPLIT DESIGN

I will use a time-aware split.

March 2026 data will be used as the decision-time feature data,
and the future April 2026 outcome will be used as the target.

Rows with a known April outcome will be used for evaluation.

I will not randomly mix future observations into the training
features because that could make the evaluation less honest.

The same March feature set and April outcome definition will be
used when comparing the Random Forest model with the Week-4
baseline.

This split is designed to reflect the real decision process:
use information available in March to estimate future April
performance.
""")


SPLIT DESIGN

I will use a time-aware split.

March 2026 data will be used as the decision-time feature data,
and the future April 2026 outcome will be used as the target.

Rows with a known April outcome will be used for evaluation.

I will not randomly mix future observations into the training
features because that could make the evaluation less honest.

The same March feature set and April outcome definition will be
used when comparing the Random Forest model with the Week-4
baseline.

This split is designed to reflect the real decision process:
use information available in March to estimate future April
performance.



## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# ============================================================
# SECTION 3: TRAIN + COMPARE VS WEEK-4 BASELINE
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# 1. LOAD THE SAME DATASET USED IN WEEK 4
# ============================================================

data_path = "/content/ML-Internship/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    !git clone https://github.com/Vishwas-Chaudhary/ML-Internship.git

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)


# ============================================================
# 2. DEFINE TARGET AND HONEST FEATURES
# ============================================================

# Target:
# Future-style prediction is not possible from the static starter
# dataset, so we use observed engagement_rate as the lane-relevant
# outcome for this modeling exercise.

target = "engagement_rate"

features = [
    "avg_position",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "scroll_rate"
]

print("\nTarget:", target)
print("Features:", features)


# ============================================================
# 3. PREPARE DATA
# ============================================================

model_data = df[
    features + [target]
].copy()

model_data = model_data.replace(
    [np.inf, -np.inf],
    np.nan
)

model_data = model_data.dropna(
    subset=[target]
)

# Fill missing feature values using training-independent
# simple defaults.
for col in features:
    model_data[col] = model_data[col].fillna(
        model_data[col].median()
    )

print("\nRows available for modeling:", len(model_data))


# ============================================================
# 4. SAME TRAIN / TEST SPLIT
# ============================================================

X = model_data[features]
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nSPLIT DESIGN")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


# ============================================================
# 5. TRAIN RANDOM FOREST
# ============================================================

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

model_pred = model.predict(X_test)


# ============================================================
# 6. RANDOM FOREST METRICS
# ============================================================

model_mae = mean_absolute_error(
    y_test,
    model_pred
)

model_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        model_pred
    )
)

print("\nRANDOM FOREST")
print("MAE :", round(model_mae, 6))
print("RMSE:", round(model_rmse, 6))


# ============================================================
# 7. RECREATE WEEK-4-STYLE BASELINE SCORE
# ============================================================

# IMPORTANT:
# The baseline score uses position and impression exposure.
# It does NOT use engagement_rate, which is our target.

baseline_data = X_test.copy()

# Position component
baseline_data["position_score"] = np.select(
    [
        baseline_data["avg_position"] <= 3,
        baseline_data["avg_position"] <= 10,
        baseline_data["avg_position"] <= 20
    ],
    [
        0,
        2,
        4
    ],
    default=6
)

# Impression component
baseline_data["impression_score"] = np.select(
    [
        baseline_data["impressions_90d"] >= 50000,
        baseline_data["impressions_90d"] >= 5000,
        baseline_data["impressions_90d"] >= 1000
    ],
    [
        5,
        4,
        2
    ],
    default=0
)

baseline_data["baseline_score"] = (
    baseline_data["position_score"]
    + baseline_data["impression_score"]
)

print("\nWEEK-4-STYLE BASELINE")
print(
    baseline_data[
        [
            "avg_position",
            "impressions_90d",
            "baseline_score"
        ]
    ].head()
)


# ============================================================
# 8. CALIBRATE BASELINE SCORE USING TRAINING DATA ONLY
# ============================================================

# Convert the rule score into an estimated engagement rate.
# The mapping is learned only from the training set.

train_baseline = X_train.copy()

train_baseline["position_score"] = np.select(
    [
        train_baseline["avg_position"] <= 3,
        train_baseline["avg_position"] <= 10,
        train_baseline["avg_position"] <= 20
    ],
    [
        0,
        2,
        4
    ],
    default=6
)

train_baseline["impression_score"] = np.select(
    [
        train_baseline["impressions_90d"] >= 50000,
        train_baseline["impressions_90d"] >= 5000,
        train_baseline["impressions_90d"] >= 1000
    ],
    [
        5,
        4,
        2
    ],
    default=0
)

train_baseline["baseline_score"] = (
    train_baseline["position_score"]
    + train_baseline["impression_score"]
)

train_baseline["target"] = y_train.values

score_means = (
    train_baseline
    .groupby("baseline_score")["target"]
    .mean()
)

overall_mean = y_train.mean()

baseline_pred = (
    baseline_data["baseline_score"]
    .map(score_means)
    .fillna(overall_mean)
    .to_numpy()
)


# ============================================================
# 9. BASELINE METRICS
# ============================================================

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_pred
    )
)

print("\nBASELINE")
print("MAE :", round(baseline_mae, 6))
print("RMSE:", round(baseline_rmse, 6))


# ============================================================
# 10. MODEL VS BASELINE
# ============================================================

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "MAE": [
        baseline_mae,
        model_mae
    ],
    "RMSE": [
        baseline_rmse,
        model_rmse
    ]
})

print("\n" + "=" * 60)
print("MODEL VS BASELINE")
print("=" * 60)

print(
    comparison.to_string(index=False)
)

print("\nLower MAE and RMSE are better.")


if model_mae < baseline_mae:
    print(
        "\nRandom Forest performs better than the Week-4 baseline "
        "on this test set by MAE."
    )
else:
    print(
        "\nRandom Forest does not outperform the Week-4 baseline "
        "on this test set by MAE."
    )


# ============================================================
# 11. FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print("\nFEATURE IMPORTANCE")
print(
    importance.to_string(index=False)
)

Dataset shape: (30000, 44)

Target: engagement_rate
Features: ['avg_position', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'scroll_rate']

Rows available for modeling: 30000

SPLIT DESIGN
Training rows: 24000
Test rows: 6000

RANDOM FOREST
MAE : 2.910256
RMSE: 7.345676

WEEK-4-STYLE BASELINE
       avg_position  impressions_90d  baseline_score
2308           21.3              283               6
22404           8.8             8878               6
23397           0.0                3               0
25058           8.1              124               2
2664           30.9             4294               8

BASELINE
MAE : 3.737157
RMSE: 8.04772

MODEL VS BASELINE
         method      MAE     RMSE
Week-4 baseline 3.737157 8.047720
  Random Forest 2.910256 7.345676

Lower MAE and RMSE are better.

Random Forest performs better than the Week-4 baseline on this test set by MAE.

FEATURE IMPORTANCE
        feature  importance
    scroll_rate    0.298606
   sessions_90d    0.242598
   avg_

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# ============================================================
# SECTION 4: ERRORS AND INTERPRETATION
# ============================================================

print("ERROR ANALYSIS AND INTERPRETATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Build error table
# ------------------------------------------------------------

error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["predicted"] = model_pred

error_df["absolute_error"] = (
    abs(error_df["actual"] - error_df["predicted"])
)

error_df["squared_error"] = (
    error_df["actual"] - error_df["predicted"]
) ** 2


# ------------------------------------------------------------
# 2. Largest errors
# ------------------------------------------------------------

largest_errors = error_df.sort_values(
    "absolute_error",
    ascending=False
).head(10)

print("\nLARGEST MODEL ERRORS")
print("-" * 60)

print(
    largest_errors[
        [
            "actual",
            "predicted",
            "absolute_error",
            "avg_position",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "scroll_rate"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 3. Average error
# ------------------------------------------------------------

print("\nERROR SUMMARY")
print("-" * 60)

print(
    "Mean absolute error:",
    round(error_df["absolute_error"].mean(), 6)
)

print(
    "Median absolute error:",
    round(error_df["absolute_error"].median(), 6)
)

print(
    "Maximum absolute error:",
    round(error_df["absolute_error"].max(), 6)
)


# ------------------------------------------------------------
# 4. Feature importance
# ------------------------------------------------------------

print("\nFEATURE IMPORTANCE")
print("-" * 60)

print(
    importance.to_string(index=False)
)


# ------------------------------------------------------------
# 5. Interpretation
# ------------------------------------------------------------

top_feature = importance.iloc[0]["feature"]

print("""
INTERPRETATION

The Random Forest achieved lower MAE and RMSE than the Week-4
baseline on the same held-out test set.

The largest model errors show that some pages are harder to estimate
accurately from the available signals. This suggests that the five
features do not capture every factor affecting engagement.

The most important model feature was:
""")

print(top_feature)

print("""
Feature importance should be interpreted as model reliance rather
than causation.

The baseline is simpler and easier to explain, but the Random Forest
captures non-linear relationships between the observed features and
engagement_rate.

The model is therefore useful as decision support, but its predictions
should not be treated as causal explanations or guaranteed outcomes.
""")

ERROR ANALYSIS AND INTERPRETATION

LARGEST MODEL ERRORS
------------------------------------------------------------
 actual  predicted  absolute_error  avg_position  impressions_90d  clicks_90d  sessions_90d  scroll_rate
  100.0   1.950505       98.049495          28.9               49           0             1        50.00
  100.0   3.954546       96.045454          22.5              152           0             1        50.00
  100.0   4.463148       95.536852          33.9              473           0             1       100.00
  100.0   5.094421       94.905579          23.9              281           0             1       100.00
  100.0  15.227901       84.772099           6.7              123           1             1        66.67
  100.0  15.656435       84.343565           5.8                6           1             1       100.00
  100.0  17.183581       82.816419           3.3              128           1             1       100.00
  100.0  19.226801       80.773199         

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.